In [1]:
# recommendation system using collaborative and content based filtering
# task 4 - codsoft internship

import math

# some movies with genres
movies = {
    1:  {"title": "The Dark Knight",         "genres": ["Action", "Crime", "Drama"]},
    2:  {"title": "Inception",               "genres": ["Action", "Sci-Fi", "Thriller"]},
    3:  {"title": "The Shawshank Redemption","genres": ["Drama"]},
    4:  {"title": "Interstellar",            "genres": ["Sci-Fi", "Drama", "Adventure"]},
    5:  {"title": "Avengers Endgame",        "genres": ["Action", "Sci-Fi", "Adventure"]},
    6:  {"title": "The Godfather",           "genres": ["Crime", "Drama"]},
    7:  {"title": "Pulp Fiction",            "genres": ["Crime", "Drama", "Thriller"]},
    8:  {"title": "The Matrix",              "genres": ["Sci-Fi", "Action"]},
    9:  {"title": "Forrest Gump",            "genres": ["Drama", "Romance"]},
    10: {"title": "Silence of the Lambs",    "genres": ["Crime", "Thriller"]},
}

# user ratings out of 5
ratings = {
    "Alice": {1:5, 2:4, 3:3, 5:4, 8:5},
    "Bob":   {2:5, 4:5, 5:3, 8:4},
    "Carol": {3:5, 6:5, 7:4, 9:5, 10:4},
    "Dave":  {1:4, 6:5, 7:5, 10:5},
    "Eve":   {2:3, 4:5, 8:4},
}

def cosine_sim(user1_ratings, user2_ratings):
    # find movies both users rated
    common = set(user1_ratings.keys()) & set(user2_ratings.keys())
    if not common:
        return 0

    dot_product = sum(user1_ratings[m] * user2_ratings[m] for m in common)
    norm1 = math.sqrt(sum(v**2 for v in user1_ratings.values()))
    norm2 = math.sqrt(sum(v**2 for v in user2_ratings.values()))

    if norm1 == 0 or norm2 == 0:
        return 0
    return dot_product / (norm1 * norm2)

# collaborative filtering - find similar users and recommend their movies
def collaborative_filter(user, n=5):
    if user not in ratings:
        print("User not found")
        return []

    user_rated = set(ratings[user].keys())

    # get similarity with all other users
    similarities = {}
    for other_user in ratings:
        if other_user != user:
            sim = cosine_sim(ratings[user], ratings[other_user])
            similarities[other_user] = sim

    # calculate predicted scores for movies user hasn't seen
    scores = {}
    sim_sum = {}

    for other_user, sim in similarities.items():
        if sim <= 0:
            continue
        for movie_id, rating in ratings[other_user].items():
            if movie_id not in user_rated:
                if movie_id not in scores:
                    scores[movie_id] = 0
                    sim_sum[movie_id] = 0
                scores[movie_id] += sim * rating
                sim_sum[movie_id] += sim

    # normalize
    predicted = {}
    for mid in scores:
        predicted[mid] = scores[mid] / sim_sum[mid]

    # sort and return top n
    sorted_movies = sorted(predicted.items(), key=lambda x: x[1], reverse=True)
    return sorted_movies[:n]

# content based - recommend based on genres user liked
def content_based_filter(user, n=5):
    if user not in ratings:
        return []

    user_rated = set(ratings[user].keys())

    # find genres user likes (rated 4 or above)
    liked_genres = set()
    for mid, r in ratings[user].items():
        if r >= 4:
            liked_genres.update(movies[mid]["genres"])

    if not liked_genres:
        return []

    # score unseen movies by genre match
    scores = {}
    for mid, info in movies.items():
        if mid not in user_rated:
            movie_genres = set(info["genres"])
            # jaccard similarity
            intersection = len(liked_genres & movie_genres)
            union = len(liked_genres | movie_genres)
            scores[mid] = intersection / union if union > 0 else 0

    sorted_movies = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return sorted_movies[:n]

def main():
    print("Movie Recommendation System - Task 4")
    print("=====================================")
    print("Available users:", ", ".join(ratings.keys()))

    user = input("\nEnter username: ").strip()

    if user not in ratings:
        print("User not found! Using Alice as example")
        user = "Alice"

    print(f"\nMovies {user} has already rated:")
    for mid, r in ratings[user].items():
        print(f"  {movies[mid]['title']} - {r}/5")

    print("\n--- Collaborative Filtering Recommendations ---")
    collab = collaborative_filter(user)
    if collab:
        for i, (mid, score) in enumerate(collab, 1):
            print(f"{i}. {movies[mid]['title']} (predicted score: {score:.2f})")
    else:
        print("No recommendations found")

    print("\n--- Content Based Recommendations ---")
    content = content_based_filter(user)
    if content:
        for i, (mid, score) in enumerate(content, 1):
            genres = ", ".join(movies[mid]["genres"])
            print(f"{i}. {movies[mid]['title']} - [{genres}] (score: {score:.2f})")
    else:
        print("No recommendations found")

if __name__ == "__main__":
    main()

Movie Recommendation System - Task 4
Available users: Alice, Bob, Carol, Dave, Eve

Enter username: Alice

Movies Alice has already rated:
  The Dark Knight - 5/5
  Inception - 4/5
  The Shawshank Redemption - 3/5
  Avengers Endgame - 4/5
  The Matrix - 5/5

--- Collaborative Filtering Recommendations ---
1. Interstellar (predicted score: 5.00)
2. The Godfather (predicted score: 5.00)
3. Forrest Gump (predicted score: 5.00)
4. Pulp Fiction (predicted score: 4.59)
5. Silence of the Lambs (predicted score: 4.59)

--- Content Based Recommendations ---
1. Interstellar - [Sci-Fi, Drama, Adventure] (score: 0.50)
2. Pulp Fiction - [Crime, Drama, Thriller] (score: 0.50)
3. The Godfather - [Crime, Drama] (score: 0.33)
4. Silence of the Lambs - [Crime, Thriller] (score: 0.33)
5. Forrest Gump - [Drama, Romance] (score: 0.14)
